In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece regex

import itertools, json, re
from typing import Any, Dict, Tuple
import regex
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto')
model.eval()

In [ ]:
TOOLS = {
 'searchHomes': {'properties': {'location': {'type':'string'}, 'maxPrice': {'type':'number'}, 'minRating': {'type':'number'}}, 'required': []},
 'getHomeDetails': {'properties': {'homeId': {'type':'string'}, 'homeName': {'type':'string'}}, 'required': []},
 'getUserBookings': {'properties': {}, 'required': []},
 'createBooking': {'properties': {'homeId': {'type':'string'}, 'homeName': {'type':'string'}, 'checkIn': {'type':'string'}, 'checkOut': {'type':'string'}, 'guests': {'type':'integer'}}, 'required': []},
 'cancelBooking': {'properties': {'bookingId': {'type':'string'}, 'homeName': {'type':'string'}, 'reason': {'type':'string','enum':['Change of travel plans','Found alternative accommodation','Medical or personal emergency','Accidental / duplicate booking','Host requested cancellation','Other solid reason']}, 'reasonDetails': {'type':'string'}}, 'required':['reason','reasonDetails']},
 'manageFavourites': {'properties': {'action': {'type':'string','enum':['list','add','remove']}, 'homeId': {'type':'string'}, 'homeName': {'type':'string'}}, 'required':['action']},
 'predictDynamicPricing': {'properties': {'location': {'type':'string'}, 'category': {'type':'string'}, 'guests': {'type':'integer'}, 'amenities': {'type':'array','items':{'type':'string'}}}, 'required':['location']}
}

# Deliberately excluded because they are not public HavenTo fields:
# createBooking.location, cancelBooking.cancelAll

JSON_STRING = r'"(?:\\(?:["\\/bfnrt]|u[0-9a-fA-F]{4})|[^"\\\x00-\x1F])*"'
JSON_NUMBER = r'-?(?:0|[1-9][0-9]*)(?:\.[0-9]+)?(?:[eE][+-]?[0-9]+)?'
JSON_INTEGER = r'-?(?:0|[1-9][0-9]*)'

def lit(s): return re.escape(json.dumps(s, ensure_ascii=False, separators=(',',':')))

def value_re(spec):
    if 'enum' in spec: return '(?:'+'|'.join(lit(x) for x in spec['enum'])+')'
    if spec['type']=='string': return JSON_STRING
    if spec['type']=='number': return JSON_NUMBER
    if spec['type']=='integer': return JSON_INTEGER
    if spec['type']=='array':
        item=JSON_STRING
        return rf'\[(?:{item}(?:,{item})*)?\]'
    raise ValueError(spec)

def args_re(schema):
    props=schema['properties']; required=set(schema['required']); keys=list(props)
    if not keys: return r'\{\}'
    alts=[]
    for n in range(len(keys)+1):
        for chosen in itertools.permutations(keys,n):
            if not required.issubset(chosen): continue
            fields=[lit(k)+':'+value_re(props[k]) for k in chosen]
            alts.append(r'\{'+','.join(fields)+r'\}')
    return '(?:'+'|'.join(sorted(set(alts),key=len,reverse=True))+')'

patterns=[]
for name,schema in TOOLS.items():
    patterns.append(r'\{"name":'+lit(name)+r',"arguments":'+args_re(schema)+r'\}')
GRAMMAR=regex.compile('(?:'+'|'.join(patterns)+')')

def accepts_complete(text): return GRAMMAR.fullmatch(text) is not None
def accepts_prefix(text): return GRAMMAR.fullmatch(text, partial=True) is not None

assert accepts_complete('{"name":"searchHomes","arguments":{}}')
assert accepts_complete('{"name":"searchHomes","arguments":{"maxPrice":2000}}')
assert not accepts_prefix('{"name":"searchHomes","arguments":{"price":2000}}')
assert not accepts_prefix('{"name":"cancelBooking","arguments":{"reason":"Found another place"}')
assert not accepts_prefix('[TOOL_CALLS]{"name":"searchHomes","arguments":{}}')
print('Grammar sanity checks: PASS')

In [ ]:
class HavenToCFGLogitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer, prompt_len, eos_token_id):
        self.tokenizer=tokenizer
        self.prompt_len=prompt_len
        self.eos_token_id=eos_token_id
        self.vocab_size=len(tokenizer)
        self.token_cache={}
        self.prefix_cache={}

    def token_text(self, token_id):
        if token_id not in self.token_cache:
            self.token_cache[token_id]=self.tokenizer.decode([token_id], skip_special_tokens=False, clean_up_tokenization_spaces=False)
        return self.token_cache[token_id]

    def prefix_text(self, input_ids):
        ids=input_ids[0,self.prompt_len:].tolist()
        return self.tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)

    def __call__(self, input_ids, scores):
        if input_ids.shape[0] != 1: raise ValueError('Batch size 1 only')
        prefix=self.prefix_text(input_ids)
        if accepts_complete(prefix):
            out=torch.full_like(scores,float('-inf'))
            out[0,self.eos_token_id]=scores[0,self.eos_token_id]
            return out
        scores=scores.clone()
        scores[0,self.eos_token_id]=float('-inf')
        allowed=torch.zeros(self.vocab_size,dtype=torch.bool,device=scores.device)
        for tid in range(self.vocab_size):
            text=self.token_text(tid)
            if not text: continue
            key=(prefix,tid)
            ok=self.prefix_cache.get(key)
            if ok is None:
                ok=accepts_prefix(prefix+text)
                self.prefix_cache[key]=ok
            if ok: allowed[tid]=True
        if not bool(allowed.any()): raise RuntimeError(f'CFG dead-end at {prefix!r}')
        scores[0,~allowed]=float('-inf')
        return scores

def prompt_for(user_text):
    return f'''You are HavenTo's tool router.
Convert the user's request into exactly ONE tool call.
Output ONLY the JSON object. No Markdown, no ```json, no [TOOL_CALLS], no array, no explanation, and no text before or after it.
Use only public HavenTo tool names and public argument names. Use exact enum values.

User request:
{user_text}

JSON tool call:
'''

def generate_tool_call(user_text, max_new_tokens=256):
    prompt=prompt_for(user_text)
    inputs=tokenizer(prompt,return_tensors='pt',add_special_tokens=True)
    device=model.get_input_embeddings().weight.device
    input_ids=inputs['input_ids'].to(device)
    attention_mask=inputs['attention_mask'].to(device)
    prompt_len=input_ids.shape[1]
    processor=HavenToCFGLogitsProcessor(tokenizer,prompt_len,tokenizer.eos_token_id)
    output=model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, do_sample=False, logits_processor=[processor], eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id, use_cache=True)
    raw=tokenizer.decode(output[0,prompt_len:],skip_special_tokens=True,clean_up_tokenization_spaces=False).strip()
    if not raw: raise ValueError('No tool call generated')
    obj=json.loads(raw)
    if set(obj)!={'name','arguments'} or obj['name'] not in TOOLS or not isinstance(obj['arguments'],dict): raise ValueError(f'Invalid tool call: {raw!r}')
    schema=TOOLS[obj['name']]
    unknown=set(obj['arguments'])-set(schema['properties'])
    if unknown: raise ValueError(f'Unknown argument(s): {sorted(unknown)}')
    missing=[k for k in schema['required'] if k not in obj['arguments']]
    if missing: raise ValueError(f'Missing required argument(s): {missing}')
    for k,v in obj['arguments'].items():
        spec=schema['properties'][k]
        if spec['type']=='string' and not isinstance(v,str): raise ValueError(k+' must be string')
        if spec['type']=='number' and (not isinstance(v,(int,float)) or isinstance(v,bool)): raise ValueError(k+' must be number')
        if spec['type']=='integer' and (not isinstance(v,int) or isinstance(v,bool)): raise ValueError(k+' must be integer')
        if spec['type']=='array' and not isinstance(v,list): raise ValueError(k+' must be array')
        if 'enum' in spec and v not in spec['enum']: raise ValueError(k+' has invalid enum value')
    if not accepts_complete(raw): raise ValueError(f'Outside grammar: {raw!r}')
    return obj

TEST_PROMPTS=['suggest me some homes','suggest me some homes in Bijnor','show me cheap homes under 2000','find highly rated homes','show my bookings','what is the price of Home ABC','cancel my booking because I found another place; I booked the alternative accommodation yesterday','show my favourites','add Home ABC to my favourites','predict the dynamic price for homes in Bijnor']

for user_text in TEST_PROMPTS:
    print('\nUSER:',user_text)
    try: print('TOOL CALL:',json.dumps(generate_tool_call(user_text),ensure_ascii=False))
    except Exception as e: print('ERROR:',type(e).__name__,e)

# The key fix is logits_processor=[processor]. There is intentionally no JSON extraction.
# The CFG controls syntax/schema shape; backend validation must still enforce auth, IDs, DB state, availability, and business rules.